# Smart Home Energy Forecasting
# Basic LSTM model for next-minute power prediction

This notebook uses the same household power dataset and the same target variable as the preprocessing notebook. The key fix before training is that the scaler is fit only on the training data to avoid time-series leakage.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Load the raw dataset.
data_path = Path("../data/household_power_consumption.txt")
if not data_path.exists():
    data_path = Path("data/household_power_consumption.txt")

if not data_path.exists():
    raise FileNotFoundError("Dataset not found at ../data/household_power_consumption.txt or data/household_power_consumption.txt")

df = pd.read_csv(data_path, sep=';', na_values='?')

expected_columns = ["Date", "Time", "Global_active_power", "Global_reactive_power", "Voltage", "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]
missing_columns = [col for col in expected_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())


Dataset loaded successfully.
Shape: (2075259, 9)
Columns: ['Date', 'Time', 'Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']


### 1. Check the real structure and missing-value pattern

The inspection notebook showed that the file uses Date, Time, and Global_active_power. We validate those assumptions before training.


In [2]:
print("Missing values by column:")
print(df.isnull().sum())
print("\nGlobal_active_power missing count:", df["Global_active_power"].isnull().sum())


Missing values by column:
Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64

Global_active_power missing count: 25979


### 2. Convert Date and Time to a timestamp and sort by time

We combine the date and time columns into one timestamp and keep the data in chronological order before modeling.


In [3]:
if "Date" in df.columns and "Time" in df.columns:
    df["timestamp"] = pd.to_datetime(df["Date"] + " " + df["Time"])
    df = df.sort_values("timestamp").reset_index(drop=True)
    print(df[["Date", "Time", "timestamp"]].head())
else:
    raise ValueError("The dataset does not contain the expected Date and Time columns.")


         Date      Time           timestamp
0  16/12/2006  17:24:00 2006-12-16 17:24:00
1  16/12/2006  17:25:00 2006-12-16 17:25:00
2  16/12/2006  17:26:00 2006-12-16 17:26:00
3  16/12/2006  17:27:00 2006-12-16 17:27:00
4  16/12/2006  17:28:00 2006-12-16 17:28:00


/var/folders/t1/pd3xbl0s3mv037q4k1zws1n40000gn/T/ipykernel_19840/3943573696.py:2: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["timestamp"] = pd.to_datetime(df["Date"] + " " + df["Time"])


### 3. Fill missing values and select the target

The inspection notebook showed 25,979 missing values in Global_active_power. A simple beginner-friendly fix is to fill those missing values before training.


In [4]:
missing_before = df["Global_active_power"].isnull().sum()
print("Missing values before fill:", missing_before)

df["Global_active_power"] = df["Global_active_power"].ffill().bfill()
print("Missing values after fill:", df["Global_active_power"].isnull().sum())

# Use a smaller real subset for this beginner-friendly first run.
# This keeps training manageable while preserving the actual data pattern.
target = df["Global_active_power"].astype(float).head(30000)

print("Target length:", len(target))

Missing values before fill: 25979
Missing values after fill: 0
Target length: 30000


### 4. Fix the leakage issue: fit the scaler on training data only

This is the important correction before model training. The scaler must be fitted on the training portion only. If it is fit on the full dataset, the model sees future information and the evaluation is too optimistic.


In [5]:
sequence_length = 60
train_size = int(len(target) * 0.8)

train_target = target.iloc[:train_size]
test_target = target.iloc[train_size:]

scaler = MinMaxScaler()
train_target_scaled = scaler.fit_transform(train_target.to_numpy().reshape(-1, 1)).reshape(-1)
test_target_scaled = scaler.transform(test_target.to_numpy().reshape(-1, 1)).reshape(-1)

print("Train target length:", len(train_target))
print("Test target length:", len(test_target))
print("Train scaled sample:", train_target_scaled[:5].tolist())
print("Test scaled sample:", test_target_scaled[:5].tolist())


Train target length: 24000
Test target length: 6000
Train scaled sample: [0.4499888118147237, 0.577981651376147, 0.5795479973148356, 0.5811143432535244, 0.38845379279480874]
Test scaled sample: [0.01946744238084583, 0.019691206086372794, 0.01946744238084583, 0.019243678675318864, 0.019019914969791905]


### 5. Build the LSTM input sequences

Each input sequence uses the previous 60 minutes to predict the next minute. The LSTM receives shape (samples, 60, 1), and the target is a single value for each sample.


In [6]:
X_train = np.array([train_target_scaled[i : i + sequence_length] for i in range(len(train_target_scaled) - sequence_length)])
y_train = train_target_scaled[sequence_length:][:, None]

X_test = np.array([test_target_scaled[i : i + sequence_length] for i in range(len(test_target_scaled) - sequence_length)])
y_test = test_target_scaled[sequence_length:][:, None]

X_train = X_train.reshape(-1, sequence_length, 1)
X_test = X_test.reshape(-1, sequence_length, 1)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


X_train shape: (23940, 60, 1)
y_train shape: (23940, 1)
X_test shape: (5940, 60, 1)
y_test shape: (5940, 1)


### 6. Build a simple LSTM model

This is a small initial model for a first run. It uses a simple recurrent layer followed by a dense output layer for one value prediction.


In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

model = Sequential([
    LSTM(32, input_shape=(sequence_length, 1)),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
print(model.summary())


/Users/abinav/EMEA-AI-Training/.venv/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         4,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,385 (17.13 KB)

 Trainable params: 4,385 (17.13 KB)

 Non-trainable params: 0 (0.00 B)

None


### 7. Train the model and evaluate it on the chronological test set

The model is trained on the earlier sequences and evaluated on the later sequences to preserve the time order.


In [8]:
history = model.fit(
    X_train,
    y_train,
    epochs=2,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

loss, mae_scaled = model.evaluate(X_test, y_test, verbose=0)
print("Test loss (scaled units):", loss)
print("Test MAE (scaled units):", mae_scaled)

predicted_scaled = model.predict(X_test, verbose=0).reshape(-1)
predicted_original = scaler.inverse_transform(predicted_scaled.reshape(-1, 1)).reshape(-1)
true_original = scaler.inverse_transform(y_test.reshape(-1, 1)).reshape(-1)

mae_original = np.mean(np.abs(true_original - predicted_original))
print("Test MAE in original power units:", mae_original)
print("First 5 true values:", true_original[:5].tolist())
print("First 5 predicted values:", predicted_original[:5].tolist())


Epoch 1/2


  1/749 ━━━━━━━━━━━━━━━━━━━━ 23:42 2s/step - loss: 0.0317 - mae: 0.1424

  7/749 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.0243 - mae: 0.1147  

 13/749 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.0161 - mae: 0.0916

 20/749 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 0.0127 - mae: 0.0828

...

740/749 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0029 - mae: 0.0288

746/749 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0030 - mae: 0.0288

749/749 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - loss: 0.0029 - mae: 0.0288 - val_loss: 8.3649e-04 - val_mae: 0.0156


Test loss (scaled units): 0.0008364876266568899
Test MAE (scaled units): 0.015625329688191414


Test MAE in original power units: 0.1396591502262286
First 5 true values: [0.466, 0.464, 0.492, 0.546, 0.542]
First 5 predicted values: [0.5397929549217224, 0.5405052900314331, 0.5402492880821228, 0.5528834462165833, 0.5830481052398682]
